# Food Delivery Time Prediction
Complete project covering:
- Data Preprocessing & Feature Engineering
- K-Means Clustering
- Hierarchical Clustering
- Neural Network Classification
- Model Evaluation & Insights


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, classification_report, confusion_matrix
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

# Load Dataset
df = pd.read_csv("Food_Delivery_Time_Prediction 2.csv")
df.head()


## Data Cleaning and Feature Engineering

In [ ]:

# Missing values
df = df.drop_duplicates()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(exclude="object").columns:
    df[col] = df[col].fillna(df[col].median())

# Rush Hour Feature
df["Order_Time"] = pd.to_datetime(df["Order_Time"])
df["Rush_Hour"] = df["Order_Time"].dt.hour.isin([8,9,13,14,18,19,20]).astype(int)

# Target Variable
threshold = df["Delivery_Time"].median()
df["Delivery_Status"] = (df["Delivery_Time"] > threshold).astype(int)

df.head()


## Encoding and Scaling

In [ ]:

cat_cols = df.select_dtypes(include="object").columns

encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str))
    encoders[c] = le

X = df.drop(["Delivery_Status"], axis=1)
y = df["Delivery_Status"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## K-Means Clustering

In [ ]:

wcss = []
for k in range(1,11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(6,4))
plt.plot(range(1,11), wcss, marker="o")
plt.title("Elbow Method")
plt.xlabel("Clusters")
plt.ylabel("WCSS")
plt.show()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
df["KMeans_Cluster"] = clusters
df["KMeans_Cluster"].value_counts()


## Hierarchical Clustering

In [ ]:

Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(10,5))
dendrogram(Z)
plt.title("Hierarchical Clustering Dendrogram")
plt.show()

hc = AgglomerativeClustering(n_clusters=3)
df["HC_Cluster"] = hc.fit_predict(X_scaled)
df["HC_Cluster"].value_counts()


## Neural Network Classification

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=16,
    verbose=1
)

loss, acc = model.evaluate(X_test, y_test)
print("Neural Network Accuracy:", acc)


## Logistic Regression Baseline

In [ ]:

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

preds = lr.predict(X_test)

print(classification_report(y_test, preds))


## Business Insights


1. Analyze cluster characteristics to identify delivery patterns.
2. High-traffic and adverse-weather clusters generally show longer delivery times.
3. Neural Networks can capture nonlinear interactions between traffic, weather, distance, and ratings.
4. Use clustering results for route optimization and workforce planning.
5. Prioritize rush-hour resource allocation to reduce delays.
